# Week 2 ID2221

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

#AQE
#spark.conf.set("spark.sql.adaptive.enabled", "false")


26/09/18 00:29:53 WARN Utils: Your hostname, DESKTOP-HNJ2ASM resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/18 00:29:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/moh/ID2/spark-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/moh/.ivy2/cache
The jars for the packages stored in: /home/moh/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7b7b069c-6be2-4a7f-90db-69ed7cd84c46;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 133ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   

In [2]:
integrated_taxi_trips = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")
integrated_taxi_trips_by_date = spark.read.format("delta").load("delta/integrated_taxi_trips_by_date")



In [3]:
#function to fairly measure time

import time

def measure(df, runs=3):
    """Run the query fully, several times, return the median time in ms."""
    times = []
    for i in range(runs):
        start = time.time()
        df.write.format("noop").mode("overwrite").save()   # forces full computation, writes nothing
        times.append((time.time() - start) * 1000)
    times.sort()
    return f"{round(times[len(times) // 2])} ms"   # median

1. Monthly taxi demand for each taxi zone

In [4]:
Q1 = (integrated_taxi_trips
          .select("pu_zone", "tpep_pickup_datetime")
          .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
          .groupBy(["pu_zone", "month"])
          .count())

# Uncomment this line if you are doing latency measurements
# By default, spark is lazy and only calculates when the results are requested

# result.collect()

#Q1.show(5)

measure(Q1)

'1414 ms'

In [5]:
#Q1.explain("formatted")

2. Average trip distance under different weather conditions

In [6]:
# Here we look at the average trip distance if there is rain, or no rain.

Q2 = (
    integrated_taxi_trips
          .select("trip_distance", "weather_prcp")
          .dropna()
          .withColumn("is_raining", F.col("weather_prcp") != 0.0)
          .groupBy("is_raining")
          .avg("trip_distance")
          .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
          .drop("weather_prcp", "avg(trip_distance)")
    )

measure(Q2)

'735 ms'

In [7]:
#Q2.show()

3. Relationship between air quality and taxi demand

In [8]:
# Get the hourly taxi demand associated to the average air quality measurement
Q3 = (
    integrated_taxi_trips
        .select("tpep_pickup_datetime","air_q_sample_measurement")
        # A bunch of lines do not have any air quality data, they are removed
        .dropna()
        .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .groupBy("hour")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.avg("air_q_sample_measurement").alias("air_quality")
        )
)

# Calculate the correlation between air quality and taxi demand
correlation = Q3.stat.corr(
    "air_quality",
    "taxi_demand"
)

print(correlation) # Result: corr = -0.017 -> No correlation 

[Stage 40:===================================================>    (11 + 1) / 12]

-0.016985133728975532


In [9]:
measure(Q3)

'902 ms'

4. Taxi zones with the largest variation in demand under different weather conditions

In [10]:
Q4 = (
    integrated_taxi_trips
        .select("pu_zone", "weather_prcp")
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        
)

Q4.show()

+--------------------+------------+----------+
|             pu_zone|weather_prcp|is_raining|
+--------------------+------------+----------+
|        West Village|         0.0|     false|
|    Garment District|         0.0|     false|
|            Gramercy|         0.0|     false|
|Penn Station/Madi...|         0.0|     false|
|        Midtown East|         0.0|     false|
|        Clinton East|         0.0|     false|
|      Midtown Center|         0.0|     false|
|      Midtown Center|         0.0|     false|
|         Murray Hill|         0.0|     false|
|            Kips Bay|         0.0|     false|
|Penn Station/Madi...|         0.0|     false|
|Penn Station/Madi...|         0.0|     false|
|  World Trade Center|         0.0|     false|
|     Lower East Side|         0.0|     false|
|Upper West Side S...|         0.0|     false|
|        West Village|         0.0|     false|
|     Lenox Hill East|         0.0|     false|
|     Lenox Hill West|         0.0|     false|
|Upper West S

5. Peak travel hours for each day of the week

In [11]:
Q5 = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5 = (
    Q5
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)


Q5.show()

+-----------+----+------+
|day_of_week|hour| count|
+-----------+----+------+
|     Friday|  18|110876|
|     Monday|  18| 87453|
|   Saturday|  19|103639|
|     Sunday|   0| 84325|
|   Thursday|  18|126855|
|    Tuesday|  18|106166|
|  Wednesday|  18|117895|
+-----------+----+------+



6. Monthly trends in taxi demand

In [12]:
Q6 = (integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("month")
    .count()
 )

window = Window.orderBy("month")

# Calculate the difference of number of trips between the current month and the previous month
Q6 = (
    Q6
    .withColumn("previous_count", F.lag("count").over(window))
    .withColumn("difference", F.expr("count - previous_count"))
)

#Q6.show()

In [13]:
measure(Q6)

'930 ms'

## Caching

In [14]:

spark.catalog.clearCache()

print("Q1 no cached:", measure(Q1))
print("Q4 no cached:", measure(Q4))
print("Q5 no cached:", measure(Q5))
print("Q6 no cached:", measure(Q6))

base = integrated_taxi_trips.select("pu_zone", "tpep_pickup_datetime", "weather_prcp").cache()
base.count()  #action to force spark

Q1_cached = (base
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy(["pu_zone", "month"])
    .count())

window = Window.orderBy("month")
Q6_cached = (base
    .select("tpep_pickup_datetime")
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("month")
    .count()
    .withColumn("previous_count", F.lag("count").over(window))
    .withColumn("difference", F.expr("count - previous_count")))

Q4_cached = (
    base
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        .drop("tpep_pickup_datetime")
        
)

window2 = Window.partitionBy("day_of_week").orderBy(F.desc("count"))
Q5_cached = (
    base
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count().withColumn("rank", F.row_number().over(window2))
    .filter(F.col("rank") == 1)
    .drop("rank")
)






#this should show "InMemoryTableScan" if the cache is actually being hit
#Q1_cached.explain("formatted")
#Q6_cached.explain("formatted")

print("Q1 cached:", measure(Q1_cached))
print("Q4 cached:", measure(Q4_cached))
print("Q5 cached:", measure(Q5_cached))
print("Q6 cached:", measure(Q6_cached))

spark.catalog.clearCache()

Q1 no cached: 1161 ms
Q4 no cached: 701 ms


Q5 no cached: 1267 ms
Q6 no cached: 795 ms


Q1 cached: 751 ms
Q4 cached: 342 ms


Q5 cached: 978 ms
Q6 cached: 429 ms


## Partition Pruning

In [15]:
#average fare in Manhattan

integrated_taxi_trips = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")

#prunes on the borough table
version_a = integrated_taxi_trips.filter(F.col("pu_borough") == "Manhattan").groupBy("pu_borough").agg(F.round(F.avg("fare_amount"), 4).alias("avg_fare"))

#prunes on time table
version_b = integrated_taxi_trips_by_date.filter(F.col("pu_borough") == "Manhattan").groupBy("pu_borough").agg(F.round(F.avg("fare_amount"), 4).alias("avg_fare"))

version_a.explain("formatted")
version_b.explain("formatted")


print("A (borough table):", measure(version_a))
print("B (time table):", measure(version_b))
# prove they're the same
assert version_a.exceptAll(version_b).count() == 0
assert version_b.exceptAll(version_a).count() == 0

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [fare_amount#5462, pu_borough#5496]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/moh/ID2/delta/integrated_taxi_trips_by_borough]
PartitionFilters: [isnotnull(pu_borough#5496), (pu_borough#5496 = Manhattan)]
ReadSchema: struct<fare_amount:double>

(2) HashAggregate
Input [2]: [fare_amount#5462, pu_borough#5496]
Keys [1]: [pu_borough#5496]
Functions [1]: [partial_avg(fare_amount#5462)]
Aggregate Attributes [2]: [sum#5868, count#5869L]
Results [3]: [pu_borough#5496, sum#5870, count#5871L]

(3) Exchange
Input [3]: [pu_borough#5496, sum#5870, count#5871L]
Arguments: hashpartitioning(pu_borough#5496, 200), ENSURE_REQUIREMENTS, [plan_id=4543]

(4) HashAggregate
Input [3]: [pu_borough#5496, sum#5870, count#5871L]
Keys [1]: [pu_borough#5496]
Functions [1]: [avg(fare_amount#5462)]
Aggregate Attributes [1]: [avg(fa

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/moh/ID2/delta/integrated_taxi_trips_by_date]
PushedFilters: [IsNotNull(pu_borough), EqualTo(pu_borough,Manhattan)]
ReadSchema: struct<fare_amount:double,pu_borough:string>

(2) Filter
Input [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]
Condition : (isnotnull(pu_borough#266) AND (pu_borough#266 = Manhattan))

(3) Project
Output [2]: [fare_amount#232, pu_borough#266]
Input [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]

(4) HashAggregate
Input [2]: [fare_amount#232, pu_borough#266]
Keys [1]: [pu_borough#266]
Functions [1]: [partial_avg(fare_amount#232)]
Aggregate Attributes [2]: [sum#6237, count#6238L]
Results [3]: [pu_borough#266, 

B (time table): 1090 ms


## Broadcast joins vs. Shuffle joins

In [16]:
import pyspark.sql.functions as F

taxi_trips_all = (
    spark.read.format("delta").load("delta/taxi_trips_01")
    .unionByName(spark.read.format("delta").load("delta/taxi_trips_02"))
    .unionByName(spark.read.format("delta").load("delta/taxi_trips_03"))
)

weather = spark.read.format("delta").load("delta/weather")

#AQE can force broadcast. disable it
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


# shuffle join
Q2_shuffle = (
    taxi_trips_all
    .withColumn("weather_time", F.date_trunc("hour", F.col("tpep_pickup_datetime")))
    .join(
        weather,
        on=F.col("weather_time") == F.col("timestamp"),
        how="inner"
    )
    .select("trip_distance", "prcp")
    .dropna(subset=["prcp"])
    .withColumn("is_raining", F.col("prcp") != 0.0)
    .groupBy("is_raining")
    .avg("trip_distance")
    .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
)

Q2_shuffle.explain(mode="formatted")
print("Shuffle Q2 Execution Time:", measure(Q2_shuffle))


# broadcast join
Q2_broadcast = (
    taxi_trips_all
    .withColumn("weather_time", F.date_trunc("hour", F.col("tpep_pickup_datetime")))
    .join(
        F.broadcast(weather), # The optimization hint
        on=F.col("weather_time") == F.col("timestamp"),
        how="inner"
    )
    .select("trip_distance", "prcp")
    .dropna(subset=["prcp"])
    .withColumn("is_raining", F.col("prcp") != 0.0)
    .groupBy("is_raining")
    .avg("trip_distance")
    .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
)

Q2_broadcast.explain(mode="formatted")
print("Broadcast Q2 Execution Time:", measure(Q2_broadcast))

# 3. Restore default Spark configurations 
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

== Physical Plan ==
* Project (26)
+- * HashAggregate (25)
   +- Exchange (24)
      +- * HashAggregate (23)
         +- * Project (22)
            +- * SortMergeJoin Inner (21)
               :- * Sort (15)
               :  +- Exchange (14)
               :     +- Union (13)
               :        :- * Project (4)
               :        :  +- * Filter (3)
               :        :     +- * ColumnarToRow (2)
               :        :        +- Scan parquet  (1)
               :        :- * Project (8)
               :        :  +- * Filter (7)
               :        :     +- * ColumnarToRow (6)
               :        :        +- Scan parquet  (5)
               :        +- * Project (12)
               :           +- * Filter (11)
               :              +- * ColumnarToRow (10)
               :                 +- Scan parquet  (9)
               +- * Sort (20)
                  +- Exchange (19)
                     +- * Filter (18)
                        +- * ColumnarToRow 

Shuffle Q2 Execution Time: 3146 ms
== Physical Plan ==
* Project (23)
+- * HashAggregate (22)
   +- Exchange (21)
      +- * HashAggregate (20)
         +- * Project (19)
            +- * BroadcastHashJoin Inner BuildRight (18)
               :- Union (13)
               :  :- * Project (4)
               :  :  +- * Filter (3)
               :  :     +- * ColumnarToRow (2)
               :  :        +- Scan parquet  (1)
               :  :- * Project (8)
               :  :  +- * Filter (7)
               :  :     +- * ColumnarToRow (6)
               :  :        +- Scan parquet  (5)
               :  +- * Project (12)
               :     +- * Filter (11)
               :        +- * ColumnarToRow (10)
               :           +- Scan parquet  (9)
               +- BroadcastExchange (17)
                  +- * Filter (16)
                     +- * ColumnarToRow (15)
                        +- Scan parquet  (14)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#7305, trip_distanc

[Stage 392:==============================================>        (33 + 6) / 39]

Broadcast Q2 Execution Time: 1846 ms


## AQE: on vs off

In [17]:
#AQE on
Q5_on = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5_on = (
    Q5_on
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("Q5 execution time with AQE on:", measure(Q5_on))
Q5_on.explain("formatted")


Q5 execution time with AQE on: 1003 ms
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- Filter (12)
      +- Window (11)
         +- WindowGroupLimit (10)
            +- Sort (9)
               +- Exchange (8)
                  +- WindowGroupLimit (7)
                     +- Sort (6)
                        +- HashAggregate (5)
                           +- Exchange (4)
                              +- HashAggregate (3)
                                 +- Project (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#5453, pu_borough#5496]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/moh/ID2/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<tpep_pickup_datetime:timestamp>

(2) Project
Output [2]: [hour(tpep_pickup_datetime#5453, Some(Europe/Stockholm)) AS hour#13583, date_format(tpep_pickup_datetime#5453, EEEE, Some(Europe/Stockholm)) AS day_of_week#13586]
Input [2]: [tpep_pickup_datetime

In [18]:
#AQE off
spark.conf.set("spark.sql.adaptive.enabled", "false")

Q5_off = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5_off = (
    Q5_off
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("Q5 execution time with AQE off:", measure(Q5_off))
Q5_off.explain("formatted")


spark.conf.set("spark.sql.adaptive.enabled", "true")

Q5 execution time with AQE off: 1386 ms
== Physical Plan ==
* Project (14)
+- * Filter (13)
   +- Window (12)
      +- WindowGroupLimit (11)
         +- * Sort (10)
            +- Exchange (9)
               +- WindowGroupLimit (8)
                  +- * Sort (7)
                     +- * HashAggregate (6)
                        +- Exchange (5)
                           +- * HashAggregate (4)
                              +- * Project (3)
                                 +- * ColumnarToRow (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#5453, pu_borough#5496]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/moh/ID2/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<tpep_pickup_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [tpep_pickup_datetime#5453, pu_borough#5496]

(3) Project [codegen id : 1]
Output [2]: [hour(tpep_pickup_datetime#5453, Some(Europe/Stockholm)) AS hour#14043, d